<a href="https://colab.research.google.com/github/jphysycs/PhD/blob/main/An%C3%A1lise_explorat%C3%B3ria_com_(Efici%C3%AAncia).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import re
import requests
import pandas as pd
from datetime import datetime


def ler_rpt_github(url):
    """
    Lê um arquivo .RPT do GitHub e retorna um DataFrame
    com a tabela de picos.
    """

    # Converte URL do GitHub para RAW
    url = url.replace(
        "https://github.com/",
        "https://raw.githubusercontent.com/"
    ).replace("/blob/", "/")

    texto_response = requests.get(url)
    texto_response.raise_for_status()

    linhas = texto_response.text.splitlines()

    dados = []

    for linha in linhas:

        linha = linha.strip()

        # Split the line into parts
        partes = linha.split()

        # Skip lines that are too short to be a peak entry
        # A peak entry without 'M'/'m' has 7 numeric fields.
        # If it has 'M'/'m', it has 8 fields.
        if len(partes) < 7:
            continue

        # Determine if 'Tipo' (M/m) is present
        tipo_present = False
        if partes[0] == 'M' or partes[0] == 'm':
            tipo_present = True

        try:
            current_data = {}
            if tipo_present:
                # Line starts with 'M' or 'm', e.g., 'M 1 0.79 ...'
                if len(partes) < 8: # If 'M'/'m' is present, we expect at least 8 parts
                    continue
                current_data["Tipo"] = partes[0]
                current_data["Pico"] = int(partes[1])
                current_data["Energia_keV"] = float(partes[2])
                current_data["Resolucao_keV"] = float(partes[3])
                current_data["Background"] = float(partes[4])
                current_data["Contagens"] = float(partes[5])
                current_data["Contagens_1s"] = float(partes[6])
                current_data["Canal"] = float(partes[7])
            else:
                # Line starts directly with the peak number, e.g., '3 9.92 ...'
                current_data["Tipo"] = '' # No 'M' or 'm' prefix
                current_data["Pico"] = int(partes[0])
                current_data["Energia_keV"] = float(partes[1])
                current_data["Resolucao_keV"] = float(partes[2])
                current_data["Background"] = float(partes[3])
                current_data["Contagens"] = float(partes[4])
                current_data["Contagens_1s"] = float(partes[5])
                current_data["Canal"] = float(partes[6])

            dados.append(current_data)

        except ValueError:
            # This will catch lines that do not conform to the expected numeric format,
            # effectively filtering out non-peak header/footer lines.
            pass

    return pd.DataFrame(dados)

# Define the base URL for the repository folder up to 'Eficiencia/'
base_repo_url = "https://raw.githubusercontent.com/jphysycs/PhD/refs/heads/main/Eficiencia/"

# Define valid folders
valid_folders = ["BG", "P0", "P1", "P2", "P3", "P4", "P5", "P6", "P7"]

# Ask the user for the folder name
print(f"Please choose a folder from the following options: {', '.join(valid_folders)}")
folder_name = input("Enter the folder name: ")

# Validate the folder name
while folder_name not in valid_folders:
    print("Invalid folder name. Please choose from the provided options.")
    folder_name = input("Enter the folder name: ")

# Ask the user for the RPT filename
filename = input("Please enter the .RPT filename (e.g., 137CS148S15.RPT): ")

# Construct the full URL for the selected file
url = base_repo_url + folder_name + "/" + filename

# Fetch the raw content for the new URL and store in the global 'texto' variable
response = requests.get(url)
response.raise_for_status()
texto = response.text

# Process the raw content for date/time
lines = texto.splitlines()
for i in range(4):
    print(lines[i])

# Extract relevant lines
date_time_str = lines[1].strip()
measurement_time_str = lines[2].strip()
real_time_str = lines[3].strip()

# Parse the date and time
date_time_obj = datetime.strptime(date_time_str, '%d/%m/%Y %H:%M:%S')

# Extract numeric values for time in seconds
measurement_seconds = float(measurement_time_str.split()[0])
real_seconds = float(real_time_str.split()[0])

# Create a dictionary for the DataFrame
data_to_df = {
    'DateTime': [date_time_obj],
    'Measurement_Time_seconds': [measurement_seconds],
    'Real_Time_seconds': [real_seconds]
}

# Create the DataFrame
df_times = pd.DataFrame(data_to_df)

# Display the DataFrame
display(df_times)

# Get the peak DataFrame using the new URL
df = ler_rpt_github(url)

Please choose a folder from the following options: BG, P0, P1, P2, P3, P4, P5, P6, P7
Enter the folder name: P0
Please enter the .RPT filename (e.g., 137CS148S15.RPT): 137CS148S15.RPT
C:\GENIE2K\CAMFILES\Semmler\JP\P0\137CS148S15.CNF            
04/02/2025 11:29:22   
     600.00 seconds
     646.67 seconds


,DateTime,Measurement_Time_seconds,Real_Time_seconds
0,2025-02-04 11:29:22,600.0,646.67


In [27]:
df

,Tipo,Pico,Energia_keV,Resolucao_keV,Background,Contagens,Contagens_1s,Canal
0,,1,2.18,0.75,4727.0,3680.785,3.35,29.41
1,,2,22.25,0.87,12772.0,7950.518,2.36,109.61
2,,3,32.13,0.82,30435.0,238274.344,0.23,149.09
3,,4,36.39,0.73,19825.0,58551.664,0.57,166.11
4,,5,74.98,0.71,9522.0,643.902,23.13,320.34
5,,6,661.70,1.23,2707.0,726685.063,0.12,2665.21
6,M,7,693.61,1.38,448.0,363.070,7.80,2792.71
7,m,8,697.91,1.38,215.0,104.265,14.85,2809.93
8,,9,1323.37,1.66,11.0,417.704,5.14,5309.61


In [28]:
df_times

,DateTime,Measurement_Time_seconds,Real_Time_seconds
0,2025-02-04 11:29:22,600.0,646.67


In [30]:
import re
import requests
import pandas as pd
import numpy as np

from datetime import datetime

from google.colab import files


print(
    "Selecione a planilha Excel utilizada para o cálculo:"
)

uploaded_excel = files.upload()


excel_filename = list(
    uploaded_excel.keys()
)[0]


print(
    "\nPlanilha carregada:"
)

print(
    excel_filename
)

df_planilha = pd.read_excel(
    excel_filename,
    sheet_name=0,
    header=1 # Specify that the header is in the second row (0-indexed)
)


print(
    "\nColunas encontradas na planilha:"
)

print(
    df_planilha.columns.tolist()
)

df_planilha = df_planilha.rename(
    columns={
        "Nuclídeo":
        "Nuclideo",
        "Data / Hora":
        "Data_Referencia",
        "Atividade, Bq":
        "Atividade_Bq",
        "T1/2, d":
        "Meia_Vida_dias",
        "Aquisição / Hora":
        "Data_Aquisicao",
        "LT, s":
        "Live_Time_s",
        "LT, d":
        "Live_Time_dias",
        "TT, s":
        "Tempo_Total_s",
        "DT, %":
        "Dead_Time_pct",
        "Posição":
        "Posicao",
        "Eg, keV":
        "Energia_keV",
        "Ig ":
        "I_gamma",
        "unc.":
        "Incerteza_I_gamma",
        "unc., %": # Original column name in the spreadsheet
        "Incerteza_Atividade_pct", # Renamed to avoid conflict and be more descriptive
        "unc., %.1":
        "Incerteza_D_pct_from_unc_1", # Renamed to resolve ambiguity with 'unc., %.2'
        "tD, d":
        "Tempo_Decorrido_dias",
        "D":
        "Fator_Decadimento",
        "unc., %.2":
        "Incerteza_D_pct_from_unc_2", # Renamed to resolve ambiguity with 'unc., %.1'
        "C":
        "Fator_C",
        "unc., %.3":
        "Incerteza_C_pct",
        "Eficiência":
        "Eficiencia",
        "Area":
        "Area"
    }
)


print(
    "\nColunas após renomeação:"
)

for i, coluna in enumerate(
    df_planilha.columns
):

    print(
        i,
        coluna
    )


base_repo_url = (
    "https://raw.githubusercontent.com/"
    "jphysycs/PhD/refs/heads/main/Eficiencia/"
)


valid_folders = [
    "BG",
    "P0",
    "P1",
    "P2",
    "P3",
    "P4",
    "P5",
    "P6",
    "P7"
]


print(
    "\nPastas disponíveis:"
)

print(
    ", ".join(
        valid_folders
    )
)


folder_name = input(
    "\nDigite o nome da pasta: "
)


while (
    folder_name
    not in
    valid_folders
):

    print(
        "Pasta inválida."
    )

    folder_name = input(
        "Digite novamente: "
    )


filename = input(
    "\nDigite o nome do arquivo .RPT "
    "(ex.: 133Ba146S15.RPT): "
)


url = (
    base_repo_url
    +
    folder_name
    +
    "/"
    +
    filename
)


print(
    "\nURL do arquivo:"
)

print(
    url
)

def ler_rpt_github(url):
    """
    Lê um arquivo .RPT do GitHub.

    Retorna um DataFrame contendo:

    Tipo
    Pico
    Energia_keV
    Resolucao_keV
    Background
    Contagens
    Contagens_1s
    Canal
    """

    url = url.replace(
        "https://github.com/",
        "https://raw.githubusercontent.com/"
    ).replace(
        "/blob/",
        "/"
    )

    response = requests.get(
        url
    )

    response.raise_for_status()

    linhas = response.text.splitlines()


    dados = []

    for linha in linhas:

        linha = linha.strip()


        if not linha:

            continue


        partes = linha.split()


        if len(partes) < 7:

            continue


        tipo_present = (
            partes[0] == "M"
            or
            partes[0] == "m"
        )


        try:

            current_data = {}

            if tipo_present:

                if len(partes) < 8:

                    continue


                current_data["Tipo"] = (
                    partes[0]
                )

                current_data["Pico"] = int(
                    partes[1]
                )

                current_data["Energia_keV"] = float(
                    partes[2]
                )

                current_data["Resolucao_keV"] = float(
                    partes[3]
                )

                current_data["Background"] = float(
                    partes[4]
                )

                current_data["Contagens"] = float(
                    partes[5]
                )

                current_data["Contagens_1s"] = float(
                    partes[6]
                )

                current_data["Canal"] = float(
                    partes[7]
                )

            else:

                current_data["Tipo"] = ""

                current_data["Pico"] = int(
                    partes[0]
                )

                current_data["Energia_keV"] = float(
                    partes[1]
                )

                current_data["Resolucao_keV"] = float(
                    partes[2]
                )

                current_data["Background"] = float(
                    partes[3]
                )

                current_data["Contagens"] = float(
                    partes[4]
                )

                current_data["Contagens_1s"] = float(
                    partes[5]
                )

                current_data["Canal"] = float(
                    partes[6]
                )


            dados.append(
                current_data
            )


        except ValueError:

            continue


    return pd.DataFrame(
        dados
    )


response = requests.get(
    url
)

response.raise_for_status()


texto = response.text


lines = texto.splitlines()


print(
    "\nPrimeiras linhas do arquivo .RPT:"
)


for i in range(

    min(
        4,
        len(lines)
    )

):

    print(
        lines[i]
    )


date_time_str = (

    lines[1].strip()

)


measurement_time_str = (

    lines[2].strip()

)


real_time_str = (

    lines[3].strip()

)


date_time_obj = datetime.strptime(

    date_time_str,

    "%d/%m/%Y %H:%M:%S"

)


measurement_seconds = float(

    measurement_time_str.split()[0]

)


real_seconds = float(

    real_time_str.split()[0]

)


df_times = pd.DataFrame({

    "DateTime":

    [date_time_obj],

    "Measurement_Time_seconds":

    [measurement_seconds],

    "Real_Time_seconds":

    [real_seconds]

})


print(
    "\nDados obtidos do .RPT:"
)

display(
    df_times
)


df_rpt = ler_rpt_github(
    url
)


print(
    "\nPicos encontrados no .RPT:"
)

display(
    df_rpt
)


filename_upper = filename.upper()


if "133BA" in filename_upper:

    nuclideo = "Ba-133"

elif "137CS" in filename_upper:

    nuclideo = "Cs-137"

elif "60CO" in filename_upper:

    nuclideo = "Co-60"

elif "152EU" in filename_upper:

    nuclideo = "Eu-152"

elif "166HO" in filename_upper:

    nuclideo = "Ho-166m"


elif "170TM" in filename_upper:

    nuclideo = "Tm-170"

elif "155EU" in filename_upper:

    nuclideo = "Eu-155"


else:

    nuclideo = None


print(
    "\nNuclídeo identificado:"
)

print(
    nuclideo
)


match_fabricante = re.search(

    r"(\d+S\d+)",

    filename_upper

)


if match_fabricante:

    fabricante = (

        match_fabricante.group(1)

    )

else:

    fabricante = None


print(
    "Fabricante identificado:"
)

print(
    fabricante
)


df_ref = df_planilha.copy()


if nuclideo is not None:

    df_ref = df_ref[

        df_ref[
            "Nuclideo"
        ]

        ==

        nuclideo

    ]


if fabricante is not None:

    df_ref = df_ref[

        df_ref[
            "Fabricante"
        ]

        ==

        fabricante

    ]


print(
    "\nDados correspondentes encontrados na planilha:"
)

display(
    df_ref
)


if len(df_ref) == 0:

    raise ValueError(

        "Nenhuma linha correspondente foi encontrada "
        "na planilha para o nuclídeo e fabricante."

    )


live_time_planilha = (

    df_ref[
        "Live_Time_s"
    ]

    .iloc[0]

)


if pd.isna(
    live_time_planilha
):

    K = measurement_seconds

else:

    K = float(
        live_time_planilha
    )


def encontrar_pico_planilha(

    energia_rpt,

    tolerancia=2.0

):

    """

    Encontra na planilha a linha cuja energia
    está mais próxima da energia encontrada no .RPT.

    Tolerância padrão: ±2 keV.

    """


    diferencas = np.abs(

        df_ref[
            "Energia_keV"
        ]

        -

        energia_rpt

    )


    indice = diferencas.idxmin()


    diferenca_minima = (

        diferencas.loc[
            indice
        ]

    )


    if (

        diferenca_minima

        <=

        tolerancia

    ):

        return (

            df_ref.loc[
                indice
            ],

            diferenca_minima

        )


    return (
        None,
        None
    )


resultados = []


for _, pico in df_rpt.iterrows():


    energia_rpt = (

        pico[
            "Energia_keV"
        ]

    )


    linha_planilha, diferenca = (

        encontrar_pico_planilha(

            energia_rpt,

            tolerancia=2.0

        )

    )


    if linha_planilha is None:

        continue


    D = float(

        linha_planilha[
            "Atividade_Bq"
        ]

    )


    E = float(

        linha_planilha[
            "Incerteza_Atividade_pct"
        ]

    )


    Q = float(

        linha_planilha[
            "I_gamma"
        ]

    )


    R = float(

        linha_planilha[
            "Incerteza_I_gamma"
        ]

    )


    S = (

        100

        *

        R

        /

        Q

    )


    T = float(

        pico[
            "Contagens"
        ]

    )


    # U is now sourced from the .RPT file, 'Contagens_1s' column
    U = float(
        pico[
            "Contagens_1s"
        ]
    )


    data_referencia = (

        linha_planilha[
            "Data_Referencia"
        ]

    )


    V = (

        date_time_obj

        -

        data_referencia

    ).total_seconds() / 86400


    meia_vida_dias = float(

        linha_planilha[
            "Meia_Vida_dias"
        ]

    )


    H = np.log(2) / meia_vida_dias


    W = np.exp(

        -H

        *

        V

    )


    # Using Incerteza_D_pct_from_unc_1 for 'I'. Please confirm if this is the correct uncertainty to use.
    I = (

        float(

            linha_planilha[
                "Incerteza_D_pct_from_unc_1"
            ]

        )

        if "Incerteza_D_pct_from_unc_1" in df_planilha.columns

        else 0

    )


    X = (

        V

        *

        H

        *

        I

    )


    L = K / 86400


    if (

        H * L

    ) != 0:

        Y = (

            1

            -

            np.exp(

                -H * L

            )

        ) / (

            H * L

        )

    else:

        Y = 1.0


    Z = abs(

        (

            np.exp(

                -H * L

            )

            /

            Y

        )

        -

        1

    ) * I


    K = float(
        K
    )


    eficiencia = (

        T

        /

        (

            D

            *

            Q

            *

            W

            *

            Y

            *

            K

        )

    )


    AA = eficiencia


    incerteza_eficiencia = (

        AA

        *

        np.sqrt(

            U**2

            +

            (

                E / 2

            )**2

            +

            S**2

            +

            X**2

            +

            Z**2

        )

        /

        100

    )


    incerteza_relativa = (

        incerteza_eficiencia

        /

        eficiencia

        *

        100

    )


    resultados.append({

        "Nuclideo":
        nuclideo,

        "Fabricante":
        fabricante,

        "Pico":
        pico[
            "Pico"
        ],

        "Energia_RPT_keV":
        energia_rpt,

        "Energia_Planilha_keV":
        linha_planilha[
            "Energia_keV"
        ],

        "Diferenca_Energia_keV":
        diferenca,

        "Area_T":
        T,

        "Incerteza_Area_U_pct":
        U,

        "Atividade_D_Bq":
        D,

        "Incerteza_Atividade_E_pct":
        E,

        "I_gamma_Q":
        Q,

        "Incerteza_I_gamma_R":
        R,

        "Incerteza_I_gamma_S_pct":
        S,

        "Tempo_Decorrido_V_dias":
        V,

        "Fator_Decadimento_W":
        W,

        "Incerteza_W_X_pct":
        X,

        "Fator_Aquisicao_Y":
        Y,

        "Incerteza_Y_Z_pct":
        Z,

        "Live_Time_K_s":
        K,

        "Eficiencia_AA":
        AA,

        "Incerteza_Eficiencia":
        incerteza_eficiencia,

        "Incerteza_Relativa_pct":
        incerteza_relativa

    })


df_resultado = pd.DataFrame(
    resultados
)


print(
    "\n"
    + "=" * 70
)

print(
    "RESULTADO FINAL DO CÁLCULO DE EFICIÊNCIA"
)

print(
    "=" * 70
)


display(
    df_resultado
)


print(
    "\nResultados principais:"
)


colunas_principais = [

    "Pico",

    "Energia_RPT_keV",

    "Energia_Planilha_keV",

    "Area_T",

    "Eficiencia_AA",

    "Incerteza_Eficiencia",

    "Incerteza_Relativa_pct"

]


display(

    df_resultado[
        colunas_principais
    ]

)


nome_saida = (

    "Resultado_Eficiencia_"

    +
    nuclideo.replace(
        "-",
        ""
    )

    +
    "_"

    +
    folder_name

    +
    ".xlsx"

)


df_resultado.to_excel(

    nome_saida,

    index=False

)


print(
    "\nArquivo de resultados gerado:"
)

print(
    nome_saida
)


files.download(
    nome_saida
)

Selecione a planilha Excel utilizada para o cálculo:


Saving EFIC_P0 - Copia.xlsx to EFIC_P0 - Copia (2).xlsx

Planilha carregada:
EFIC_P0 - Copia (2).xlsx

Colunas encontradas na planilha:
['Nuclídeo', 'Fabricante', 'Data / Hora', 'Atividade, Bq', 'unc., %', 'T1/2, d', 'unc., %.1', 'l, d-1', 'unc., %.2', 'Aquisição / Hora', 'LT, s', 'LT, d', 'TT, s', 'DT, %', 'Posição', 'Eg, keV', 'Ig ', 'unc.', 'unc., %.3', 'Area', 'unc., %.4', 'tD, d', 'D', 'unc., %.5', 'C', 'unc., %.6', 'Eficiência', 'unc..1', 'unc., %.7']

Colunas após renomeação:
0 Nuclideo
1 Fabricante
2 Data_Referencia
3 Atividade_Bq
4 Incerteza_Atividade_pct
5 Meia_Vida_dias
6 Incerteza_D_pct_from_unc_1
7 l, d-1
8 Incerteza_D_pct_from_unc_2
9 Data_Aquisicao
10 Live_Time_s
11 Live_Time_dias
12 Tempo_Total_s
13 Dead_Time_pct
14 Posicao
15 Energia_keV
16 I_gamma
17 Incerteza_I_gamma
18 Incerteza_C_pct
19 Area
20 unc., %.4
21 Tempo_Decorrido_dias
22 Fator_Decadimento
23 unc., %.5
24 Fator_C
25 unc., %.6
26 Eficiencia
27 unc..1
28 unc., %.7

Pastas disponíveis:
BG, P0, P1, P2, P3, P4,

,DateTime,Measurement_Time_seconds,Real_Time_seconds
0,2025-02-04 11:29:22,600.0,646.67



Picos encontrados no .RPT:


,Tipo,Pico,Energia_keV,Resolucao_keV,Background,Contagens,Contagens_1s,Canal
0,,1,2.18,0.75,4727.0,3680.785,3.35,29.41
1,,2,22.25,0.87,12772.0,7950.518,2.36,109.61
2,,3,32.13,0.82,30435.0,238274.344,0.23,149.09
3,,4,36.39,0.73,19825.0,58551.664,0.57,166.11
4,,5,74.98,0.71,9522.0,643.902,23.13,320.34
5,,6,661.70,1.23,2707.0,726685.063,0.12,2665.21
6,M,7,693.61,1.38,448.0,363.070,7.80,2792.71
7,m,8,697.91,1.38,215.0,104.265,14.85,2809.93
8,,9,1323.37,1.66,11.0,417.704,5.14,5309.61


...


Resultados principais:


,Pico,Energia_RPT_keV,Energia_Planilha_keV,Area_T,Eficiencia_AA,Incerteza_Eficiencia,Incerteza_Relativa_pct
0,6,661.7,661.657,726685.063,0.023281,0.000175,0.750374



Arquivo de resultados gerado:
Resultado_Eficiencia_Cs137_P0.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>